# What This Walkthrough Does

This capstone-style notebook puts the pieces together:

- explicit configuration
- folder structure
- API request
- raw output
- processed table
- manifest
- limitations

## Packages Used in This Notebook

This walkthrough uses:

- `requests` to collect data from the MediaWiki API
- `pandas` to create processed tables and save small JSON files
- `Path` to keep raw, processed, and report outputs separate
- `datetime` and `timezone` to timestamp the manifest

The point is not only to collect data, but to package the collection as a reproducible mini-project.


In [ ]:
# datetime/timezone let us create an unambiguous UTC timestamp for the manifest.
from datetime import datetime, timezone
# Path helps create output folders and file paths.
from pathlib import Path

# pandas turns extracted rows into tables and helps save small JSON records.
import pandas as pd
# requests sends the API request.
import requests


# Configuration

Configuration makes parameters visible and reviewable.

In [ ]:
config = {
    "project_name": "notebook_reproducible_api_project",
    "endpoint": "https://en.wikipedia.org/w/api.php",
    "query": "platform governance",
    "pages": 1,
    "page_size": 10,
    "outdir": "data",
}

print(config)


# Prepare Folders

In [ ]:
outdir = Path(config["outdir"])

raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

for folder in [raw_dir, processed_dir, reports_dir]:
    # mkdir(..., parents=True, exist_ok=True) creates the folder and any missing parent folders, without failing if it already exists.
    folder.mkdir(parents=True, exist_ok=True)


The folders encode workflow stages:

- `raw`: source-shaped evidence
- `processed`: analysis-shaped tables
- `reports`: manifests, audits, notes, provenance

# Collect One API Page

In [ ]:
params = {
    "action": "query",
    "list": "search",
    "srsearch": config["query"],
    "srlimit": config["page_size"],
    "format": "json",
    "formatversion": 2,
}

headers = {"User-Agent": "methodsNET-VLOP-course/1.0 reproducible project notebook"}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(config["endpoint"], params=params, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
payload = response.json()

print("Request URL:", response.url)
print("Status:", response.status_code)


# Save Raw Response

In [ ]:
raw_path = raw_dir / "notebook_project_raw_response.json"

pd.Series(
    {
        "request_url": response.url,
        "status_code": response.status_code,
        "payload": payload,
    }
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
).to_json(raw_path, indent=2)

print(raw_path)


# Process Into a Table

In [ ]:
rows = []

# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
for item in payload.get("query", {}).get("search", []):
    rows.append(
        {
            "query": config["query"],
            "pageid": item.get("pageid"),
            "title": item.get("title"),
            "timestamp": item.get("timestamp"),
            "wordcount": item.get("wordcount"),
        }
    )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
df = pd.DataFrame(rows)

processed_path = processed_dir / "notebook_project_processed.csv"
# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
df.to_csv(processed_path, index=False)

df.head()


# Write a Manifest

A manifest explains what was created and how.

In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_name": config["project_name"],
    "access_route": "MediaWiki API",
    "parameters": config,
    "raw_output": str(raw_path),
    "processed_output": str(processed_path),
    "row_count": len(df),
    "limitations": [
        "Search results are API-ranked and query-dependent.",
        "This is not a measure of all platform-governance content on Wikipedia.",
        "Only one small page of results was collected for teaching purposes.",
    ],
}

manifest_path = reports_dir / "notebook_project_manifest.json"
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(manifest).to_json(manifest_path, indent=2)

print(manifest_path)


# Exercise

Adapt the project:

1. Change the query.
2. Change the page size.
3. Add one new processed field.
4. Add one limitation to the manifest.
5. Explain what someone would need to reproduce your run.

# Key Takeaway

A reproducible collection workflow is not just code that runs. It is a small
project structure that preserves parameters, raw evidence, processed outputs,
and limitations.